# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Skills loaded: `training-honest-models` + `flyrank/flyrank-data`. Lane 2 — Refresh / Content
Opportunity Scoring (locked). Same data slice and target as W04
(`data/raw/content_refresh_anonymized.csv`, `is_declining_label`), so the Week-4 rule baseline
is a fair, frozen thing to beat.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

if Path.cwd().name == "notebooks":
    import os
    os.chdir("../..")

sys.path.insert(0, "scripts")
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k

RANDOM_STATE = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Log-scaled versions the reference feature set expects (same as scripts/01_prepare_features.py).
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

print(f"Lane slice: {len(df):,} pages, {df['client_id'].nunique()} clients")
print(f"Base rate (declining proxy): {df['is_declining_label'].mean():.3f}")

# Leakage check up front: the reference feature lists never include the label's own source columns.
assert "trend_direction" not in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
assert "trend_pct" not in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
print("Leakage check passed: trend_direction / trend_pct are not in the model feature lists.")


Lane slice: 30,000 pages, 32 clients
Base rate (declining proxy): 0.542
Leakage check passed: trend_direction / trend_pct are not in the model feature lists.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Task shape:** binary label (`is_declining_label`), used for a *ranking* problem — "which pages
first?" — the same shape W02 committed to. Per the training-honest-models menu, that's:
**a classifier's probability, evaluated at precision@K** — not accuracy, because editors only
ever look at the top of the list.

**Method progression:** start with **Logistic Regression** (readable — a coefficient per
feature, easy to sanity-check), then **Random Forest** (stronger if it earns the extra
complexity). No Gradient Boosting this round: the menu flags it "where safe," and with a
30k-row, single-snapshot lane-2 slice there isn't enough headroom over Random Forest to justify
the extra opacity — that's the "does not reward complexity alone" rule in practice, decided
before fitting anything, not after peeking at scores.

**Why not clustering or plain correlation analysis:** the deliverable is a ranked action queue
(W01/W02), not groups or a single correlation coefficient — that needs a model that outputs a
per-page score.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-grouped 80/20 holdout, fixed seed 42.** A client's pages share one editorial team, one
vertical, one CMS quirk set — a row-level split would let both train and test see pages from the
same client, so the model could partly memorize *that client's* baseline instead of learning a
signal that generalizes to a client it has never seen. That's also the design W02 committed to
("client-holdout... matches the real decision: prioritize pages for clients the system has not
memorized") and what `scripts/03_train_model.py`'s `make_client_aware_split` does — built
independently here so this notebook stays self-contained and auditable on its own.

No time-aware split: this CSV is a single trailing-90-day snapshot with no report-date column to
split on (the W03/W04 caveat still holds — a stronger capstone label needs the warehouse's
prior-window → future-window design).


In [2]:
rng = np.random.default_rng(RANDOM_STATE)
clients = df["client_id"].drop_duplicates().to_numpy()
shuffled_clients = rng.permutation(clients)
n_test_clients = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:n_test_clients])

test_mask = df["client_id"].isin(test_clients).to_numpy()
train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

print(f"Train: {len(train_idx):,} rows across {df.iloc[train_idx]['client_id'].nunique()} clients")
print(f"Test:  {len(test_idx):,} rows across {df.iloc[test_idx]['client_id'].nunique()} clients")
print(f"Train positive rate: {df.iloc[train_idx]['is_declining_label'].mean():.3f}")
print(f"Test positive rate:  {df.iloc[test_idx]['is_declining_label'].mean():.3f}")
print(
    "\nHonest flag: only 6 clients land in test, so the positive rate shifts noticeably "
    "(39.1% vs. 55.5% in train) purely from which clients got shuffled where. That's a real "
    "consequence of client-holdout with only 32 clients total, not a bug -- and it means "
    "precision@10 on this test set can swing a lot from one right/wrong pick."
)


Train: 27,675 rows across 26 clients
Test:  2,325 rows across 6 clients
Train positive rate: 0.555
Test positive rate:  0.391

Honest flag: only 6 clients land in test, so the positive rate shifts noticeably (39.1% vs. 55.5% in train) purely from which clients got shuffled where. That's a real consequence of client-holdout with only 32 clients total, not a bug -- and it means precision@10 on this test set can swing a lot from one right/wrong pick.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The W04 rule (`0.45*visibility + 0.40*ctr_gap + 0.15*depth_gap`) is **frozen** exactly as
shipped last week — recomputed here only so it can be scored on *this* test split, for a fair
same-split, same-metric comparison against the two models.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

numeric_frame = df[MODEL_NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = df[MODEL_CATEGORICAL_FEATURES].fillna("unknown").astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=MODEL_CATEGORICAL_FEATURES, dtype=float)
X = pd.concat([numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)], axis=1)
y = df["is_declining_label"]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

logistic_regression = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])
logistic_regression.fit(X_train, y_train)
lr_test_prob = logistic_regression.predict_proba(X_test)[:, 1]

random_forest = RandomForestClassifier(
    class_weight="balanced_subsample",
    max_depth=8,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
random_forest.fit(X_train, y_train)
rf_test_prob = random_forest.predict_proba(X_test)[:, 1]

print(f"Feature matrix: {X.shape[1]} columns ({len(MODEL_NUMERIC_FEATURES)} numeric + one-hot categoricals)")


Feature matrix: 52 columns (18 numeric + one-hot categoricals)


In [4]:
def percentile_rank(series: pd.Series) -> pd.Series:
    return series.rank(method="average", pct=True).fillna(0)

# --- Frozen W04 rule, recomputed only to score it on this notebook's test split ---
visible = df[df["impressions_90d"] >= 500]
tier_median_ctr = visible.groupby("position_tier")["ctr"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr).fillna(0)
df["ctr_gap"] = (df["tier_median_ctr"] - df["ctr"]).clip(lower=0)
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["ctr_gap_score"] = percentile_rank(df["ctr_gap"])
df["has_word_count"] = df["word_count"].notna() & (df["word_count"] > 0)
wc_percentile = pd.Series(0.0, index=df.index)
wc_percentile.loc[df["has_word_count"]] = percentile_rank(df.loc[df["has_word_count"], "word_count"])
df["depth_gap_score"] = np.where(df["has_word_count"], (1 - wc_percentile) * df["visibility_score"], 0.0)
df["baseline_action_score"] = (
    0.45 * df["visibility_score"] + 0.40 * df["ctr_gap_score"] + 0.15 * df["depth_gap_score"]
).clip(0, 1)

baseline_test_score = df.iloc[test_idx]["baseline_action_score"].to_numpy()


def score_row(name: str, y_true: pd.Series, scores: np.ndarray) -> dict:
    return {
        "model": name,
        "precision_at_10": precision_at_k(y_true, scores, 10),
        "precision_at_20": precision_at_k(y_true, scores, 20),
        "precision_at_50": precision_at_k(y_true, scores, 50),
        "roc_auc": roc_auc_score(y_true, scores),
        "average_precision": average_precision_score(y_true, scores),
    }


comparison = pd.DataFrame([
    score_row("baseline_w04_rule (frozen)", y_test, baseline_test_score),
    score_row("logistic_regression", y_test, lr_test_prob),
    score_row("random_forest", y_test, rf_test_prob),
])
comparison["base_rate_test"] = y_test.mean()
comparison


,model,precision_at_10,precision_at_20,precision_at_50,roc_auc,average_precision,base_rate_test
0,baseline_w04_rule (frozen),0.7,0.45,0.54,0.611675,0.503192,0.390968
1,logistic_regression,0.2,0.35,0.40,0.700291,0.521542,0.390968
2,random_forest,1.0,0.70,0.68,0.742486,0.596835,0.390968


**Reading the table honestly:**

- **Random Forest wins across the board** — precision@10/20/50 all clear both the baseline and
  the base rate, and it has the best ROC-AUC and average precision too. It's the model I'd ship.
- **Logistic Regression is the "report both" case the skill warns about**: it beats the baseline
  on ROC-AUC and average precision (better *overall* ranking separation) but **loses to the
  baseline at precision@10/20/50** — worse at the very top of a small (2,325-row, 6-client) test
  set, where a single flipped pick swings precision@10 by 10 points. That's a real result, not a
  bug to explain away: a linear model's smooth probability ranking doesn't concentrate its best
  guesses at the very top the way a tree ensemble's can on this data.
- Random Forest's win isn't "because it's bigger" — `max_depth=8` and `min_samples_leaf=25` on
  ~27.7k training rows deliberately caps it well short of memorizing individual pages; it wins
  because it captures non-linear interactions (e.g. "high volume AND poor position" as a single
  split) that a linear model can only approximate as separate additive terms.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


In [5]:
from sklearn.inspection import permutation_importance

impurity_importance = pd.Series(random_forest.feature_importances_, index=X.columns).sort_values(ascending=False).head(8)
print("Random Forest — impurity-based importance (top 8):")
print(impurity_importance)
print()

perm_result = permutation_importance(random_forest, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
permutation_importance_top = pd.Series(perm_result.importances_mean, index=X.columns).sort_values(ascending=False).head(8)
print("Random Forest — permutation importance on the test split (top 8, corroboration check):")
print(permutation_importance_top)


Random Forest — impurity-based importance (top 8):
days_with_impressions    0.144946
log_impressions_90d      0.134849
avg_position             0.120378
content_age_days         0.093963
age_tier_365+            0.038763
word_count               0.037145
log_clicks_90d           0.034658
char_count               0.033767
dtype: float64



Random Forest — permutation importance on the test split (top 8, corroboration check):
days_with_impressions    0.021333
log_impressions_90d      0.015011
log_clicks_90d           0.010968
content_age_days         0.006366
ctr                      0.006194
days_with_sessions       0.005505
scroll_rate              0.005118
impression_tier_low      0.004688
dtype: float64


**What the model leans on — and does it make sense?** Both importance methods agree on the
same top signals: `days_with_impressions`, `log_impressions_90d`, `log_clicks_90d`,
`content_age_days`, `avg_position`, `ctr`. These are exactly the observable-demand and
ranking-position signals the W04 signal audit already validated (impression consistency and
position both plausibly relate to whether a page is losing ground) — none of it is
"suspiciously perfect" (no single feature anywhere near 90%+ of the importance), and nothing
from the excluded/label-derived list (`trend_direction`, `trend_pct`) appears at all, because it
was never in the feature matrix. Two independent methods (impurity, permutation) landing on the
same handful of features is a real sanity check, not a coincidence.


In [6]:
test_frame = df.iloc[test_idx].copy().reset_index(drop=True)
test_frame["rf_probability"] = rf_test_prob
test_frame["rf_prediction"] = (rf_test_prob >= 0.5).astype(int)

false_positives = test_frame[(test_frame["rf_prediction"] == 1) & (test_frame["is_declining_label"] == 0)].sort_values("rf_probability", ascending=False)
false_negatives = test_frame[(test_frame["rf_prediction"] == 0) & (test_frame["is_declining_label"] == 1)].sort_values("rf_probability")

print(f"False positives: {len(false_positives)} of {len(test_frame)} test rows")
print(f"False negatives: {len(false_negatives)} of {len(test_frame)} test rows")
print()
cols = ["content_id", "rf_probability", "impressions_90d", "avg_position", "ctr", "trend_direction", "content_age_days"]
print("3 concrete false positives (model says declining, proxy label says no):")
print(false_positives[cols].head(3).to_string(index=False))
print()
print("3 concrete false negatives (model says fine, proxy label says declining):")
print(false_negatives[cols].head(3).to_string(index=False))


False positives: 544 of 2325 test rows
False negatives: 225 of 2325 test rows

3 concrete false positives (model says declining, proxy label says no):
          content_id  rf_probability  impressions_90d  avg_position  ctr trend_direction  content_age_days
content_f5013794ba57        0.713442              881          15.7  0.0             new               175
content_331182ca4cae        0.711645             3026          35.9  0.0              up               134
content_e354f8e518c2        0.711350             1163          23.4  0.0             new               175

3 concrete false negatives (model says fine, proxy label says declining):
          content_id  rf_probability  impressions_90d  avg_position   ctr trend_direction  content_age_days
content_28b4223f4e5f        0.080854                1           0.0  0.00            down                91
content_34b14c00f80c        0.099336                3           0.0  0.00            down               308
content_472ce7ae14c0  

**Why these are hard, in plain words:**

- **False positives** cluster on pages with real volume and a mid-page-one position but
  `trend_direction` of `new` or `up`, not `down` — the model has never seen `trend_direction`
  (by design), so from a single snapshot a *rising* page with today's low CTR looks
  structurally identical to a *declining* one with today's low CTR. Fixing this needs a
  feature the CSV can't give — the CSV is one snapshot, not a history — which is exactly the
  W03/W04 caveat about the proxy label's limits.
- **False negatives** cluster on near-invisible pages (1–3 impressions, `avg_position = 0` i.e.
  no ranking data) that are labeled declining almost by definition (any drop off a near-zero
  base). With almost every numeric signal near zero, the model has nothing to separate these
  from the (much larger) pool of equally quiet, non-declining pages — it correctly treats them
  as low-confidence rather than guessing.
- **Net read:** the model is not wrong in a suspicious way — both error types trace back to
  information genuinely absent from a single-snapshot CSV, not to a bug in the features or a
  leak in the score.


In [7]:
import json as _json

metrics = {
    "random_state": RANDOM_STATE,
    "split": {
        "strategy": "client_grouped_holdout_80_20",
        "train_rows": int(len(train_idx)), "test_rows": int(len(test_idx)),
        "train_clients": int(df.iloc[train_idx]["client_id"].nunique()),
        "test_clients": int(df.iloc[test_idx]["client_id"].nunique()),
        "train_positive_rate": float(df.iloc[train_idx]["is_declining_label"].mean()),
        "test_positive_rate": float(y_test.mean()),
    },
    "comparison": comparison.set_index("model").round(4).to_dict(orient="index"),
    "random_forest_params": {"max_depth": 8, "min_samples_leaf": 25, "n_estimators": 200},
    "top_features_impurity": impurity_importance.round(4).to_dict(),
    "top_features_permutation": permutation_importance_top.round(4).to_dict(),
    "error_counts": {"false_positives": int(len(false_positives)), "false_negatives": int(len(false_negatives)), "test_rows": int(len(test_frame))},
}
metrics_path = Path("work/outputs/w05_model_metadata.json")
metrics_path.write_text(_json.dumps(metrics, indent=2, sort_keys=True))
print(f"Wrote {metrics_path}")


Wrote work/outputs/w05_model_metadata.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
